In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import re
from datetime import datetime, timezone

PROJECT_ROOT = Path.cwd().parent
PROCESSED = PROJECT_ROOT / "data" / "processed"
PRIORITY = PROCESSED / "priority"
DIAG = PROCESSED / "diagnostics"
MAPPED = PROCESSED / "mapped"
OUT = PROCESSED / "intervention"
OUT.mkdir(parents=True, exist_ok=True)

print("OUT:", OUT)

OUT: c:\Users\Administrator\Desktop\Matric-Maths-Exam-Intelligence\data\processed\intervention


In [2]:
priority = pd.read_csv(PRIORITY / "priority_bands_final_v1.csv")
errors = pd.read_csv(DIAG / "diagnostic_errors_v1.csv")

# optional files
topic_map_path = MAPPED / "question_topic_map_exams_v1.csv"
topic_map = pd.read_csv(topic_map_path) if topic_map_path.exists() else None

print("Priority rows:", len(priority))
print("Error rows:", len(errors))
print("Topic map loaded:", topic_map is not None)
print("Priority bands:\n", priority["priority_band"].value_counts())
display(priority.head())
display(errors.head(3))

Priority rows: 10
Error rows: 273
Topic map loaded: True
Priority bands:
 priority_band
Medium    7
Lower     3
Name: count, dtype: int64


,topic,exposure_norm,subquestion_count,total_marks,exposure_score,error_pressure_per_exposure,difficulty_norm,persistence_norm,design_norm,design_marks,priority_score,priority_band
0,Trigonometry,0.701560,38.0,162.0,15.92,2.381579,0.340773,1.0,1.000000,50.0,0.6468,Medium
1,Calculus,1.000000,45.0,248.0,21.85,0.511111,0.000000,1.0,0.571429,35.0,0.5571,Medium
2,Analytical Geometry,0.524409,33.0,114.0,12.40,1.984848,0.268495,1.0,0.714286,40.0,0.5361,Medium
3,Functions & Graphs,0.625063,39.0,130.0,14.40,1.615385,0.201183,1.0,0.571429,35.0,0.5251,Medium
4,Euclidean Geometry,0.244087,17.0,67.0,6.83,2.852941,0.426649,1.0,0.714286,40.0,0.5153,Medium


,year,question_number,question_title,error_text_raw,error_text_summary,source,qualitative_severity,topic
0,2023,1.0,ALGEBRA,(a) In Q1.1.2 some candidates did not write th...,NaN,DBE diagnostic,moderate_language,Algebra & Equations
1,2023,1.0,ALGEBRA,(b) In Q1.1.3 many candidates were able to squ...,NaN,DBE diagnostic,moderate_language,Algebra & Equations
2,2023,1.0,ALGEBRA,(c) Many candidates struggled to solve the ine...,NaN,DBE diagnostic,severe_language,Algebra & Equations


In [3]:
MASTERY_CRITERIA = {
    "correct_items_required": 4,
    "items_presented": 5,
    "structure_type_match": True,
    "sittings_required": 2,
    "unseen_items_only": True,
    "time_window_days": 14,
    "tutor_policy": {
        "never_give_full_solution_first": True,
        "required_sequence": [
            "diagnostic_probe",
            "guided_intervention",
            "guided_practice",
            "independent_practice",
            "mastery_check"
        ],
        "solution_release_policy": "only_after_attempt_or_mastery_fail_review"
    },
    "locked_at": datetime.now(timezone.utc).isoformat(),
}

(OUT / "mastery_criteria_v1.json").write_text(
    json.dumps(MASTERY_CRITERIA, indent=2), encoding="utf-8"
)
print("Mastery + tutor policy locked:")
print(json.dumps(MASTERY_CRITERIA, indent=2))

Mastery + tutor policy locked:
{
  "correct_items_required": 4,
  "items_presented": 5,
  "structure_type_match": true,
  "sittings_required": 2,
  "unseen_items_only": true,
  "time_window_days": 14,
  "tutor_policy": {
    "never_give_full_solution_first": true,
    "required_sequence": [
      "diagnostic_probe",
      "guided_intervention",
      "guided_practice",
      "independent_practice",
      "mastery_check"
    ],
    "solution_release_policy": "only_after_attempt_or_mastery_fail_review"
  },
  "locked_at": "2026-09-12T21:33:31.752118+00:00"
}


In [4]:
V1_TOPICS = [
    "Trigonometry",
    "Calculus",
    "Functions & Graphs",
    "Analytical Geometry",
    "Euclidean Geometry",
]

in_scope = priority[priority["topic"].isin(V1_TOPICS)][
    ["topic", "priority_band", "priority_score"]
].copy()

# Keep only these five even if band is Lower for one of them
missing = [t for t in V1_TOPICS if t not in set(in_scope["topic"])]
if missing:
    extra = priority[priority["topic"].isin(missing)][
        ["topic", "priority_band", "priority_score"]
    ]
    in_scope = pd.concat([in_scope, extra], ignore_index=True)

in_scope = in_scope.drop_duplicates("topic").sort_values(
    "priority_score", ascending=False
).reset_index(drop=True)

print("N08 v1 forced scope:")
display(in_scope)

scoped_errors = errors.merge(in_scope, on="topic", how="inner")
print("Scoped error rows:", len(scoped_errors))
print(scoped_errors["topic"].value_counts())

N08 v1 forced scope:


,topic,priority_band,priority_score
0,Trigonometry,Medium,0.6468
1,Calculus,Medium,0.5571
2,Analytical Geometry,Medium,0.5361
3,Functions & Graphs,Medium,0.5251
4,Euclidean Geometry,Medium,0.5153


Scoped error rows: 178
topic
Trigonometry           50
Analytical Geometry    40
Functions & Graphs     39
Euclidean Geometry     32
Calculus               17
Name: count, dtype: int64


In [5]:
def rough_label(text: str) -> str:
    t = (text or "").lower()
    rules = [
        (r"sign|positive|negative|inequal", "sign_or_inequality_error"),
        (r"asymptote", "asymptote_misconception"),
        (r"inverse", "inverse_function_error"),
        (r"substitut", "incorrect_substitution"),
        (r"formula|identity", "formula_identity_error"),
        (r"diagram|reason|theorem", "geometry_reasoning_gap"),
        (r"gradient|distance|midpoint|circle", "analytical_geometry_formula_error"),
        (r"derivative|turning point|max|min|concav", "calculus_interpretation_error"),
        (r"probability|counting|tree|venn", "probability_structure_error"),
        (r"sequence|series|arithmetic|geometric", "sequence_formula_error"),
        (r"finance|interest|annuity|present|future", "finance_formula_choice_error"),
        (r"graph|sketch|plot", "graph_interpretation_error"),
    ]
    for pat, label in rules:
        if re.search(pat, t):
            return label
    return "unclassified_error"

scoped_errors = scoped_errors.copy()
scoped_errors["misconception_label"] = scoped_errors["error_text_raw"].astype(str).apply(rough_label)
scoped_errors["misconception_id"] = scoped_errors["misconception_label"]

print(scoped_errors["misconception_label"].value_counts())

misconception_label
unclassified_error                   55
geometry_reasoning_gap               23
incorrect_substitution               19
sign_or_inequality_error             18
analytical_geometry_formula_error    18
calculus_interpretation_error        12
formula_identity_error               11
graph_interpretation_error           10
asymptote_misconception               6
inverse_function_error                3
finance_formula_choice_error          2
sequence_formula_error                1
Name: count, dtype: int64


In [8]:
PATTERN_MAP = {
    "sign_or_inequality_error": "IP_BOUNDARY_CASE",
    "asymptote_misconception": "IP_REPRESENTATION_SHIFT",
    "inverse_function_error": "IP_REPRESENTATION_SHIFT",
    "incorrect_substitution": "IP_STEP_ISOLATION",
    "formula_identity_error": "IP_MISCONCEPTION_CONTRAST",
    "geometry_reasoning_gap": "IP_SELF_EXPLANATION",
    "analytical_geometry_formula_error": "IP_STEP_ISOLATION",
    "calculus_interpretation_error": "IP_STEP_ISOLATION",
    "probability_structure_error": "IP_REPRESENTATION_SHIFT",
    "sequence_formula_error": "IP_MISCONCEPTION_CONTRAST",
    "finance_formula_choice_error": "IP_MISCONCEPTION_CONTRAST",
    "graph_interpretation_error": "IP_REPRESENTATION_SHIFT",
    "unclassified_error": "IP_GRADUATED_DIFFICULTY",
}

spec = (
    scoped_errors.groupby(
        ["topic", "priority_band", "priority_score", "misconception_id", "misconception_label"],
        as_index=False
    )
    .agg(
        evidence_count=("error_text_raw", "count"),
        years_documented=("year", "nunique"),
        sample_evidence=("error_text_raw", lambda s: " | ".join(s.astype(str).head(2))),
    )
)

spec["intervention_pattern"] = (
    spec["misconception_label"].map(PATTERN_MAP).fillna("IP_GRADUATED_DIFFICULTY")
)
spec["diagnostic_signal"] = (
    "Learner response matches documented pattern: " + spec["misconception_label"]
)
spec["practice_sequence_pattern"] = (
    "diagnostic → " + spec["intervention_pattern"] + " → graduated practice → mastery check"
)

# Tutor policy (only after spec exists)
spec["tutor_sequence"] = (
    "diagnostic_probe → guided_intervention → guided_practice → "
    "independent_practice → mastery_check"
)
spec["solution_policy"] = "never_full_solution_first"
spec["mastery_check_ref"] = "mastery_criteria_v1.json"
spec["intervention_status"] = "specified"
spec["blocker_reason"] = ""

def slug(s):
    return re.sub(r"[^a-z0-9]+", "_", str(s).lower()).strip("_")

spec["intervention_id"] = (
    spec["topic"].map(slug) + "__" + spec["misconception_label"].map(slug)
)

spec = spec.sort_values(
    ["priority_score", "evidence_count"], ascending=[False, False]
).reset_index(drop=True)

display(spec.head(20))
print("Specified rows:", len(spec))
print(spec["topic"].value_counts())
print(spec["intervention_pattern"].value_counts())

,topic,priority_band,priority_score,misconception_id,misconception_label,evidence_count,years_documented,sample_evidence,intervention_pattern,diagnostic_signal,practice_sequence_pattern,tutor_sequence,solution_policy,mastery_check_ref,intervention_status,blocker_reason,intervention_id
0,Trigonometry,Medium,0.6468,unclassified_error,unclassified_error,18,3,(a) Many candidates were unable to identify th...,IP_GRADUATED_DIFFICULTY,Learner response matches documented pattern: u...,diagnostic → IP_GRADUATED_DIFFICULTY → graduat...,diagnostic_probe → guided_intervention → guide...,never_full_solution_first,mastery_criteria_v1.json,specified,,trigonometry__unclassified_error
1,Trigonometry,Medium,0.6468,graph_interpretation_error,graph_interpretation_error,8,3,(f) Many candidates did not respond to Q5.2.3 ...,IP_REPRESENTATION_SHIFT,Learner response matches documented pattern: g...,diagnostic → IP_REPRESENTATION_SHIFT → graduat...,diagnostic_probe → guided_intervention → guide...,never_full_solution_first,mastery_criteria_v1.json,specified,,trigonometry__graph_interpretation_error
2,Trigonometry,Medium,0.6468,formula_identity_error,formula_identity_error,7,3,(a) In Q7.1 many candidates attempted to use t...,IP_MISCONCEPTION_CONTRAST,Learner response matches documented pattern: f...,diagnostic → IP_MISCONCEPTION_CONTRAST → gradu...,diagnostic_probe → guided_intervention → guide...,never_full_solution_first,mastery_criteria_v1.json,specified,,trigonometry__formula_identity_error
3,Trigonometry,Medium,0.6468,incorrect_substitution,incorrect_substitution,4,2,(b) Q7.2 required candidates to analyse the di...,IP_STEP_ISOLATION,Learner response matches documented pattern: i...,diagnostic → IP_STEP_ISOLATION → graduated pra...,diagnostic_probe → guided_intervention → guide...,never_full_solution_first,mastery_criteria_v1.json,specified,,trigonometry__incorrect_substitution
4,Trigonometry,Medium,0.6468,sign_or_inequality_error,sign_or_inequality_error,4,3,"(c) Q6.3.1 is a familiar question, yet many ca...",IP_BOUNDARY_CASE,Learner response matches documented pattern: s...,diagnostic → IP_BOUNDARY_CASE → graduated prac...,diagnostic_probe → guided_intervention → guide...,never_full_solution_first,mastery_criteria_v1.json,specified,,trigonometry__sign_or_inequality_error
5,Trigonometry,Medium,0.6468,geometry_reasoning_gap,geometry_reasoning_gap,3,1,(a) In Q5.1 some candidates struggled to inter...,IP_SELF_EXPLANATION,Learner response matches documented pattern: g...,diagnostic → IP_SELF_EXPLANATION → graduated p...,diagnostic_probe → guided_intervention → guide...,never_full_solution_first,mastery_criteria_v1.json,specified,,trigonometry__geometry_reasoning_gap
6,Trigonometry,Medium,0.6468,calculus_interpretation_error,calculus_interpretation_error,2,1,(d) Q5.2 was poorly answered by many candidate...,IP_STEP_ISOLATION,Learner response matches documented pattern: c...,diagnostic → IP_STEP_ISOLATION → graduated pra...,diagnostic_probe → guided_intervention → guide...,never_full_solution_first,mastery_criteria_v1.json,specified,,trigonometry__calculus_interpretation_error
7,Trigonometry,Medium,0.6468,finance_formula_choice_error,finance_formula_choice_error,2,2,(d) When answering Q6.3.2 some candidates were...,IP_MISCONCEPTION_CONTRAST,Learner response matches documented pattern: f...,diagnostic → IP_MISCONCEPTION_CONTRAST → gradu...,diagnostic_probe → guided_intervention → guide...,never_full_solution_first,mastery_criteria_v1.json,specified,,trigonometry__finance_formula_choice_error
8,Trigonometry,Medium,0.6468,asymptote_misconception,asymptote_misconception,1,1,(b) In Q7.2 some candidates had difficulty wit...,IP_REPRESENTATION_SHIFT,Learner response matches documented pattern: a...,diagnostic → IP_REPRESENTATION_SHIFT → graduat...,diagnostic_probe → guided_intervention → guide...,never_full_solution_first,mastery_criteria_v1.json,specified,,trigonometry__asymptote_misconception
9,Trigonometry,Medium,0.6468,sequence_formula_error,sequence_for

Specified rows: 36
topic
Trigonometry           10
Functions & Graphs      8
Calculus                7
Analytical Geometry     6
Euclidean Geometry      5
Name: count, dtype: int64
intervention_pattern
IP_STEP_ISOLATION            13
IP_REPRESENTATION_SHIFT       6
IP_GRADUATED_DIFFICULTY       5
IP_MISCONCEPTION_CONTRAST     5
IP_BOUNDARY_CASE              4
IP_SELF_EXPLANATION           3
Name: count, dtype: int64


In [9]:
covered = set(spec["topic"])
blocked_topics = in_scope[~in_scope["topic"].isin(covered)].copy()
blocked_topics["intervention_status"] = "specification_blocked"
blocked_topics["blocker_reason"] = "No scoped diagnostic error rows available for this topic in N06 extract"

blocked_topics.to_csv(OUT / "intervention_blockers_v1.csv", index=False)
spec.to_csv(OUT / "intervention_specification_v1.csv", index=False)

print("Blocked topics:", len(blocked_topics))
display(blocked_topics)

Blocked topics: 0


,topic,priority_band,priority_score,intervention_status,blocker_reason


In [11]:
display(spec.sort_values("evidence_count", ascending=False).head(10)[
    ["topic", "misconception_label", "evidence_count", "intervention_pattern", "solution_policy"]
])

,topic,misconception_label,evidence_count,intervention_pattern,solution_policy
0,Trigonometry,unclassified_error,18,IP_GRADUATED_DIFFICULTY,never_full_solution_first
31,Euclidean Geometry,geometry_reasoning_gap,18,IP_SELF_EXPLANATION,never_full_solution_first
23,Functions & Graphs,unclassified_error,13,IP_GRADUATED_DIFFICULTY,never_full_solution_first
17,Analytical Geometry,analytical_geometry_formula_error,12,IP_STEP_ISOLATION,never_full_solution_first
19,Analytical Geometry,unclassified_error,10,IP_GRADUATED_DIFFICULTY,never_full_solution_first
32,Euclidean Geometry,unclassified_error,10,IP_GRADUATED_DIFFICULTY,never_full_solution_first
18,Analytical Geometry,incorrect_substitution,10,IP_STEP_ISOLATION,never_full_solution_first
24,Functions & Graphs,sign_or_inequality_error,9,IP_BOUNDARY_CASE,never_full_solution_first
1,Trigonometry,graph_interpretation_error,8,IP_REPRESENTATION_SHIFT,never_full_solution_first
2,Trigonometry,formula_identity_error,7,IP_MISCONCEPTION_CONTRAST,never_full_solution_first


In [13]:
summary = f"""# Notebook 08 Summary (v1)

## Scope
Medium/High priority topics from N07, joined to N06 diagnostic error evidence.

## Counts
- Topics in scope: {len(in_scope)}
- Specified interventions: {len(spec)}
- Blocked topics: {len(blocked_topics)}

## Pattern usage
{spec['intervention_pattern'].value_counts().to_string()}

## Mastery criteria
{json.dumps(MASTERY_CRITERIA, indent=2)}

## Limitations
- v1 is topic–misconception grain (not full skill taxonomy yet)
- No practice content authored in N08
- No fabricated misconceptions
- Priority has no High band under locked thresholds; Medium clusters drive scope
"""
(OUT / "notebook08_summary.md").write_text(summary, encoding="utf-8")
print(summary)
print("NOTEBOOK 08 v1 STARTER COMPLETE")

# Notebook 08 Summary (v1)

## Scope
Medium/High priority topics from N07, joined to N06 diagnostic error evidence.

## Counts
- Topics in scope: 5
- Specified interventions: 36
- Blocked topics: 0

## Pattern usage
intervention_pattern
IP_STEP_ISOLATION            13
IP_REPRESENTATION_SHIFT       6
IP_GRADUATED_DIFFICULTY       5
IP_MISCONCEPTION_CONTRAST     5
IP_BOUNDARY_CASE              4
IP_SELF_EXPLANATION           3

## Mastery criteria
{
  "correct_items_required": 4,
  "items_presented": 5,
  "structure_type_match": true,
  "sittings_required": 2,
  "unseen_items_only": true,
  "time_window_days": 14,
  "tutor_policy": {
    "never_give_full_solution_first": true,
    "required_sequence": [
      "diagnostic_probe",
      "guided_intervention",
      "guided_practice",
      "independent_practice",
      "mastery_check"
    ],
    "solution_release_policy": "only_after_attempt_or_mastery_fail_review"
  },
  "locked_at": "2026-09-12T21:33:31.752118+00:00"
}

## Limitations
- 